# Fine-tuning Whisper with PyTorch Lightning

In [1]:
!pip install transformers datasets lightning jiwer soundfile -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 101.5 MB/s eta 0:00:00


In [2]:
import json
from pathlib import Path

import lightning as L
import torch
import soundfile as sf
from torch.utils.data import DataLoader, Dataset
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from jiwer import wer, cer

MODEL_NAME = "openai/whisper-small"   # tiny / base / small / medium
LANGUAGE = "ukrainian"
TASK = "transcribe"

DATA_ROOT = Path("/kaggle/input/datasets/sofiiapopeniukk/audio-stt-homework-v3")
MANIFESTS = DATA_ROOT / "manifests"
CHECKPOINT = Path("./checkpoints")

BATCH_SIZE = 8
MAX_EPOCHS = 5
LR = 1e-5
NUM_WORKERS = 0

## Dataset

**Що робить:** читає маніфест (jsonl-файл де кожен рядок = один запис), 
завантажує аудіо з диску, перетворює його у вхід для Whisper.

**Whisper очікує:** аудіо 16kHz mono → `input_features` (мел-спектрограма 80×3000).
За це відповідає `processor` — ми просто передаємо йому сирі семпли.

In [5]:
class TorontoDataset(Dataset):
    def __init__(self, manifest_path, processor):
        self.processor = processor
        self.records = []
        
        with open(manifest_path, "r") as f:
            for line in f:
                item = json.loads(line)
                
                p = Path(item["audio_filepath"])
                idx = next(i for i, part in enumerate(p.parts) if part == "audio")
                relative = Path(*p.parts[idx:])
                item["audio_filepath"] = str(DATA_ROOT / relative)
                
                self.records.append(item)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        path = record["audio_filepath"]
        
        audio, sr = sf.read(path)

        # перетворюємо аудіо → мел-спектрограма
        features = self.processor(
            audio, sampling_rate=sr, return_tensors="pt"
        ).input_features.squeeze(0)   # [80, 3000]

        # перетворюємо текст → token ids
        labels = self.processor.tokenizer(
            record["text"], return_tensors="pt"
        ).input_ids.squeeze(0)

        return {"input_features": features, "labels": labels}

## Collator

**Проблема:** тексти різної довжини → `labels` різних розмірів → не можна скласти у батч.

**Рішення:** паддинг до максимальної довжини в батчі. Токени паддингу замінюємо на `-100`, 
щоб `CrossEntropyLoss` їх ігнорував (стандартна практика в HuggingFace).

In [6]:
def collate_fn(batch, pad_token_id):
    features = torch.stack([x["input_features"] for x in batch])  # [B, 80, 3000]

    max_len = max(x["labels"].shape[0] for x in batch)
    labels = torch.full((len(batch), max_len), fill_value=-100, dtype=torch.long)
    for i, x in enumerate(batch):
        seq = x["labels"]
        labels[i, :seq.shape[0]] = seq

    return {"input_features": features, "labels": labels}

## DataModule

**Lightning DataModule** — окремий клас для всього що пов'язано з даними. 
Trainer викликає `train_dataloader()` і `val_dataloader()` автоматично.

Тут ми просто склеюємо Dataset + collate_fn у DataLoader.

In [7]:
class WhisperDataModule(L.LightningDataModule):
    def __init__(self, manifests_dir: Path, processor: WhisperProcessor):
        super().__init__()
        self.manifests_dir = manifests_dir
        self.processor = processor
        self._pad_id = processor.tokenizer.pad_token_id

    def setup(self, stage=None):
        self.train_ds = TorontoDataset(self.manifests_dir / "train.jsonl",      self.processor)
        self.val_ds   = TorontoDataset(self.manifests_dir / "validation.jsonl", self.processor)

    def _collate(self, batch):
        features = torch.stack([x["input_features"] for x in batch])
        max_len = max(x["labels"].shape[0] for x in batch)
        labels = torch.full((len(batch), max_len), fill_value=-100, dtype=torch.long)
        for i, x in enumerate(batch):
            seq = x["labels"]
            labels[i, :seq.shape[0]] = seq
        return {"input_features": features, "labels": labels}

    def _loader(self, ds, shuffle):
        return DataLoader(
            ds,
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            num_workers=NUM_WORKERS,
            collate_fn=self._collate,
        )

    def train_dataloader(self): return self._loader(self.train_ds, shuffle=True)
    def val_dataloader(self):   return self._loader(self.val_ds,   shuffle=False)

## Lightning Module

**Lightning Module** — обгортка над моделлю. Визначає:
- `training_step` — що робити з одним батчем під час тренування
- `validation_step` — те саме але без градієнтів
- `configure_optimizers` — який оптимізатор використовувати

**Як Whisper рахує loss:** передаємо `input_features` (аудіо) та `labels` (текст) — 
модель сама рахує cross-entropy між своїми передбаченнями та `labels`. 
Loss повертається в `outputs.loss`.

**На валідації** — генеруємо текст і рахуємо WER/CER, щоб бачити реальну якість, 
а не просто loss.

In [8]:
class WhisperModule(L.LightningModule):
    def __init__(self, model_name: str, processor: WhisperProcessor):
        super().__init__()
        self.processor = processor
        self.model = WhisperForConditionalGeneration.from_pretrained(model_name)
        self.model.train()

        self.model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
            language=LANGUAGE, task=TASK
        )
        self.model.generation_config.suppress_tokens = []

        self._val_preds = []
        self._val_refs  = []

    def forward(self, input_features, labels):
        return self.model(input_features=input_features, labels=labels)

    def training_step(self, batch, batch_idx):
        loss = self(batch["input_features"], batch["labels"]).loss
        self.log("train/loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        # loss
        loss = self(batch["input_features"], batch["labels"]).loss
        self.log("val/loss", loss, prog_bar=True)

        # генерація тексту для WER/CER
        predicted_ids = self.model.generate(batch["input_features"])
        preds = self.processor.batch_decode(predicted_ids, skip_special_tokens=True)

        # декодуємо labels (ігноруємо -100 паддинг)
        label_ids = batch["labels"].clone()
        label_ids[label_ids == -100] = self.processor.tokenizer.pad_token_id
        refs = self.processor.batch_decode(label_ids, skip_special_tokens=True)

        self._val_preds.extend(preds)
        self._val_refs.extend(refs)

    def on_validation_epoch_end(self):
        if self._val_preds:
            self.log("val/wer", wer(self._val_refs, self._val_preds), prog_bar=True)
            self.log("val/cer", cer(self._val_refs, self._val_preds), prog_bar=True)
        self._val_preds.clear()
        self._val_refs.clear()

    def configure_optimizers(self):
        # AdamW стандартний вибір для fine-tuning трансформерів
        return torch.optim.AdamW(self.parameters(), lr=LR)

## Тренування

- запускає цикли train/val епох
- логує метрики
- зберігає найкращий чекпоінт (за `val/wer`)
- підтримує 16-bit precision (`bf16`) для економії VRAM на сучасних GPU

`ModelCheckpoint` — автоматично зберігає модель коли `val/wer` покращується.

In [10]:
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

processor = WhisperProcessor.from_pretrained(MODEL_NAME)

data_module = WhisperDataModule(MANIFESTS, processor)
model_module = WhisperModule(MODEL_NAME, processor)

checkpoint_cb = ModelCheckpoint(
    dirpath=CHECKPOINT,
    filename="whisper-toronto-{epoch:02d}-{val/wer:.3f}",
    monitor="val/wer",
    mode="min",
    save_top_k=1,
)

# зупиняємо тренування якщо val/wer не покращується 3 епохи підряд
early_stop_cb = EarlyStopping(monitor="val/wer", patience=3, mode="min")

trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,
    callbacks=[checkpoint_cb, early_stop_cb],
    # mixed precision для швидкості та економії VRAM
    precision="16-mixed", 
    log_every_n_steps=10,
    val_check_interval=0.5,
    accelerator="gpu",
    devices=1,
    strategy="auto"
)

trainer.fit(model_module, datamodule=data_module)

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, 

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

## Збереження фінальної моделі у HuggingFace форматі

In [ ]:
SAVE_DIR = Path("./whisper-toronto-finetuned")

# завантажуємо найкращий чекпоінт
best = WhisperModule.load_from_checkpoint(
    checkpoint_cb.best_model_path,
    model_name=MODEL_NAME,
    processor=processor,
)

best.model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

print(f"Модель збережена у {SAVE_DIR}")
print(f"Найкращий WER: {checkpoint_cb.best_model_score:.4f}")